# KAT-VAD v2 — exploratory probe: **text-guided multi-scale CLIP windows** (pending (ba))

**Question.** Does pooling CLIP embeddings of multi-scale frame windows, weighted by how much each window looks like
the definition set `Z`'s *anomaly* text rather than its *normal* text (AnyAnomaly's WinCLIP-based attention + LaGoVAD's
`Z`), add anything to **A3's inputs** `[x; u; p]` on DoTA-CAP-dev, and does it help the **non-ego** clips where A3 is
weakest (`REPORT_V2_RESULTS.md` §6.4)?

**Exploratory, not an amendment.** The rule is fixed in `core/tools/text_window_probe.py`'s docstring **before** the
first number. It decides only whether the idea earns a pre-registered build, and never anything about v2 (A3 stays
adopted). No training: frozen features and linear probes, as in E2(d).

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps | — |
| 1 | preflight: splits, K sources, V2-S + CLIP caches, DADA zip, CAP tar parts | no |
| 2 | **windows on the 295 K sources** (T2, whole source, stride 8): 7z shard → farm → `clip_windows frames` | **yes** |
| 3 | **windows on the 569 `dota_cap_dev` clips** (protocol B, stride 3): stream the CAP tar parts → rebuild → `clip_windows cap` | **yes** |
| 4 | stage to VM disk, run `text_window_probe` (the tool applies the rule) | no |
| 5 | record | — |

**Windows.** Non-overlapping 2×2, 3×3 and 5×5 grids over the whole frame (38 cells) + the 1×1 cell. Each cell is
squashed to 224² with CLIP's `no_center_crop` transform. The 1×1 cell **is** the existing CLIP row `x`, so every clip
is gated against its own CLIP cache (mean cos ≥ 0.99, every row ≥ 0.95). A mismatch means another transform or frame
map, and the clip is refused (lessons C2/C13).

**Rule.** GO iff transfer Δ(`+text` − `[x;u;p]`) over all two-class clips has mean ≥ +0.01 **and** CI low > 0, else
KILL. Printed, never deciding:
- Δ(`+text` − `+mean`): does *text* matter beyond multi-scale?
- Δ(`+text` − `+prior`): is it semantic or just *where in the frame*?
- Every Δ split by ego / non-ego.
- The zero-shot max-margin macro.
- The text-weight entropy (≈ 1 ⇒ the text map is flat and pools like `mean`).

**Cost.** About 39 CLIP passes per sampled frame. Step 2 ≈ 12k frames, step 3 ≈ 20k frames, plus re-streaming the
CAP tar parts the 569 clips live in. Both steps resume after a disconnect. Name every number **DoTA-CAP (n/1397)**;
DoTA-eval is never read.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # data/, cache/ (CLIP, VideoMAE, window caches)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # kat-vad/ (code), outputs/ (v2 read-outs)
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'
V1_OUTPUTS = Path(DRIVE) / 'outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'TW_GRIDS'), 'this upload predates the text-window probe -- re-upload branch v2'

ENC_S    = constants.VIDEOMAE_ENCODER_S                     # A3's stream (E2(d) pick)
DATASET  = constants.DADA_ORIGIN_DATASET                    # 'DADA2000_orig'
STRIDE   = constants.FRAME_STRIDE                           # 8: K's geometry
T2       = constants.DATA_ROOT / DATASET
CENSUS   = T2 / 'counts' / 'resolved_census.json'
ORIG     = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
T2_CLIP  = constants.CLIP_CACHE_DIR / DATASET
VIDEO    = constants.VIDEO_CACHE_DIR
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
RAW      = constants.DATA_ROOT / 'MMAU' / 'raw'
GROUPS   = ['1-10', '11', '12-42', '43', '44-62']           # CAP-DATA tar groups (dota_cap_videomae.ipynb)
BATCH_GB = constants.MMAU_STREAM_BATCH_GB
REPORTS  = constants.OUTPUT_ROOT / 'REPORTS'
CAP_REP  = REPORTS / 'dota_cap'                             # the frozen alignment
K_DIRS   = [REPORTS / 'v2_K', V1_OUTPUTS / 'REPORTS' / 'v2_K']
K_REPORT = next((d for d in K_DIRS if (d / 'k_sources.txt').is_file()), None)
assert K_REPORT is not None, f'no k_sources.txt under {K_DIRS}'
OUT      = REPORTS / 'tw_probe'
LOCAL    = Path('/content/tw')                              # VM-local NVMe, never Drive
for p in (OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name in ('T2_CLIP', 'DOTA_S1', 'ORIG', 'RAW', 'K_REPORT', 'OUT'):
    print(f'{name:9s}', globals()[name])

In [ ]:
%%bash
apt-get -qq install -y p7zip-full
pip install -q "transformers==4.56.*" av einops faiss-cpu xlrd
python -c "import transformers, av, einops, faiss, xlrd; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 2 and 3 need one'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

ENV = dict(os.environ)


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')

## 1. Preflight — every input exists before any GPU minute is spent

In [ ]:
from core.data.v2_splits import lines_sha1, load_split
from core.tools import clip_windows, dota_cap, e2d_probe, extract_video_features, text_window_probe

assert 'GO iff the transfer' in (REPO / 'core/tools/text_window_probe.py').read_text(), 'probe rule missing'
cap_dev = load_split(constants.V2_SPLIT_DOTA_CAP_DEV)                    # sha1-checked; eval never loaded
assert len(cap_dev) == 569, len(cap_dev)
k_ids = [l.strip() for l in (K_REPORT / 'k_sources.txt').read_text().splitlines() if l.strip()]
k_sha1 = json.loads((K_REPORT / 'k_sources_manifest.json').read_text())['sha1']
assert lines_sha1(k_ids) == k_sha1, 'K sources do not match their manifest'
census = json.loads(CENSUS.read_text())
assert all(v in census for v in k_ids), 'a K source has no census entry'

alignment_sha = e2d_probe.dota_cap_alignment_sha256(constants.V2_SPLITS_DIR)
assert dota_cap.file_sha256(CAP_REP / dota_cap.ALIGNMENT_JSON) == alignment_sha, 'alignment != the frozen one'
S_DOTA, S_T2 = e2d_probe.dota_video_dir(VIDEO, ENC_S), e2d_probe.t2_video_dir(VIDEO, ENC_S)
e2d_probe.check_manifest(S_DOTA, dota_cap.video_manifest(ENC_S, alignment_sha))
e2d_probe.check_manifest(S_T2, extract_video_features.build_manifest(
    ENC_S, STRIDE, constants.VIDEOMAE_WEIGHTS_SHA256[ENC_S]))
for d, ids, what in ((S_DOTA, cap_dev, 'V2-S DoTA-CAP'), (S_T2, k_ids, 'V2-S K'),
                     (DOTA_S1, cap_dev, 'CLIP DoTA_s1_ncc'), (T2_CLIP, k_ids, 'CLIP T2')):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{what}: {len(miss)} missing, e.g. {miss[:3]}'
assert (ORIG / 'DADA2000.zip').is_file(), f'no DADA2000.zip under {ORIG}'
for g in GROUPS:
    assert sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*')), f'no CAP parts for group {g}'
assert (RAW / 'cap_text_annotations.xls').is_file(), 'cap_text_annotations.xls missing'

TW_T2   = clip_windows.cache_dir(DATASET, STRIDE)
TW_DOTA = clip_windows.cache_dir(constants.DOTA_CAP_DATASET, constants.TW_DOTA_STRIDE)
done_t2 = sum((TW_T2 / f'{v}.npy').exists() for v in k_ids)
done_dota = sum((TW_DOTA / f'{v}.npy').exists() for v in cap_dev)
print(f'preflight OK | dota_cap_dev {len(cap_dev)} | K {len(k_ids)} (sha1 {k_sha1[:12]}) | '
      f'windows done: K {done_t2}/{len(k_ids)}, DoTA-CAP {done_dota}/{len(cap_dev)}')
print('caches ->', TW_T2, '|', TW_DOTA)

## 2. Windows on the K sources (GPU, resumable)

The same 7z shard → symlink farm as K / E2(d) (`p4_e2d.ipynb` §2), then `clip_windows frames` at stride 8. Every source
is gated: its 1×1 cell must reproduce `cache/clip/DADA2000_orig`, or the run raises. **Re-run this cell after a
disconnect**; finished sources are skipped.

In [ ]:
SHARD = 60
root, images = constants.DADA_ORIGIN_ROOT_DIRNAME, constants.DADA_ORIGIN_IMAGES_SUBDIR


def type_video(vid):
    t, v = vid.split('_')                      # t{type:02d}_v{video:03d}
    return int(t[1:]), int(v[1:])


started = time.monotonic()
for k in range(0, len(k_ids), SHARD):
    shard, tag = k_ids[k:k + SHARD], f'{k // SHARD:03d}'
    if all((TW_T2 / f'{v}.npy').exists() for v in shard):
        print(f'shard {tag}: already encoded, skipping')
        continue
    frames, farm = LOCAL / 'frames', LOCAL / 'flat'
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    farm.mkdir(parents=True)
    patterns = [f'{root}/{t}/{v:03d}/{images}/*' for t, v in map(type_video, shard)]
    print(f'--- shard {tag}: {len(shard)} sources, {shutil.disk_usage("/content").free / 2**30:.1f} GiB free')
    run(['7z', 'x', ORIG / 'DADA2000.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', *patterns])
    for vid in shard:                          # C26: link each source's images dir under its unique id
        t, v = type_video(vid)
        src = frames / root / str(t) / f'{v:03d}' / images
        n = len(list(src.glob('*')))
        if n != census[vid]:                   # C10: a folder is not evidence of data
            raise RuntimeError(f'{vid}: {n} images extracted, census says {census[vid]}')
        (farm / vid).symlink_to(src.resolve(), target_is_directory=True)
    shard_ids = LOCAL / f'ids_{tag}.txt'
    shard_ids.write_text('\n'.join(shard) + '\n')
    run([sys.executable, '-m', 'core.tools.clip_windows', 'frames',
         '--frames-dir', farm, '--ids-file', shard_ids, '--dataset', DATASET,
         '--clip-dir', T2_CLIP, '--stride', STRIDE, '--output-dir', TW_T2, '--device', 'auto'],
        log=LOCAL / 'tw_t2.log')
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    print(f'    encoded {sum((TW_T2 / f"{v}.npy").exists() for v in k_ids)}/{len(k_ids)} | '
          f'{(time.monotonic() - started) / 60:.1f} min')

assert all((TW_T2 / f'{v}.npy').exists() for v in k_ids), 'K window cache incomplete -- rerun this cell'
print(f'windows on all {len(k_ids)} K sources: complete (every source passed the 1x1 = CLIP-row gate)')

## 3. Windows on `dota_cap_dev` (GPU, resumable)

Only the CAP videos of the 569 dev clips are streamed. Each DoTA clip is rebuilt from the **frozen** frame map, encoded
at stride 3 (the rows of `DoTA_s1_ncc[::3]`), and gated: its 1×1 cell must reproduce those rows. Per-group reports live
in `tw_probe/`. **Re-run this cell after a disconnect.**

In [ ]:
import pandas as pd

ann = pd.read_excel(RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
align = json.loads((CAP_REP / dota_cap.ALIGNMENT_JSON).read_text())['clips']
assert all(align[v]['reason'] == 'ok' for v in cap_dev)
need_cap = {align[v]['cap_id'] for v in cap_dev if not (TW_DOTA / f'{v}.npy').exists()}
dota_ids_file = LOCAL / 'dota_cap_dev.txt'
dota_ids_file.write_text(''.join(f'{v}\n' for v in cap_dev))
free_gb = shutil.disk_usage('/content').free / 1e9
assert free_gb >= 3 * BATCH_GB, f'{free_gb:.0f} GB free on /content, need {3 * BATCH_GB:.0f} -- lower BATCH_GB'

for g in GROUPS:
    lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
    ids = sorted(c for c in need_cap if lo <= type_of[c] <= hi)
    print(f'group {g}: {len(ids)} CAP videos needed')
    if not ids:
        continue
    ids_file = LOCAL / f'cap_ids_{g}.txt'
    ids_file.write_text(''.join(f'{i}\n' for i in ids))
    parts = sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.clip_windows', 'cap',
         '--alignment', CAP_REP / dota_cap.ALIGNMENT_JSON, '--parts', *parts,
         '--work-dir', LOCAL / 'work' / g, '--dota-clip-dir', DOTA_S1,
         '--dota-ids-file', dota_ids_file, '--cap-ids-file', ids_file,
         '--report', OUT / f'tw_extract_{g}.json', '--output-dir', TW_DOTA,
         '--max-batch-gb', str(BATCH_GB), '--device', 'auto'], log=LOCAL / 'tw_cap.log')
    print(f'group {g}: {(time.time() - t0) / 60:.0f} min')

report = {}
for p in sorted(OUT.glob('tw_extract_*.json')):
    report |= json.loads(p.read_text())
missing = [v for v in cap_dev if not (TW_DOTA / f'{v}.npy').exists()]
bad = {v: report.get(v, {}).get('reason', 'not_run') for v in missing}
assert not bad, f'{len(bad)} DoTA-CAP-dev clips have no window cache: {bad}'
# per clip: every row >= DOTA_CAP_MIN_COS; the 0.99 mean is a corpus gate (clip_windows docstring)
ok = [report[v] for v in cap_dev if report.get(v, {}).get('reason') == 'ok']
if ok:
    corpus = clip_windows.corpus_gate(ok)
    print(f'windows on all {len(cap_dev)} dev clips | 1x1 corpus gate: {corpus}')
    assert corpus['reason'] == 'ok', f'window transform disagrees with DoTA_s1_ncc: {corpus}'

## 4. Probe (CPU) — the tool applies the rule

The inputs are staged to VM disk first, because reading thousands of small `.npy` files from Drive is slow. The text
anchors are encoded once with the frozen CLIP text tower (pinned revision) and saved as `tw_text.npz` beside the
read-out.

In [ ]:
stage = LOCAL / 'stage'
jobs = [(DOTA_S1, stage / 'DoTA_s1_ncc', cap_dev), (T2_CLIP, stage / 'clip_t2', k_ids),
        (S_DOTA, stage / 'video' / ENC_S / S_DOTA.name, cap_dev),
        (S_T2, stage / 'video' / ENC_S / S_T2.name, k_ids),
        (TW_DOTA, stage / 'win' / TW_DOTA.name, cap_dev), (TW_T2, stage / 'win' / TW_T2.name, k_ids)]
t0 = time.time()
for src, dst, ids in jobs:
    dst.mkdir(parents=True, exist_ok=True)
    for v in ids:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
    manifest = src / constants.VIDEO_MANIFEST_FILENAME
    if manifest.exists():
        shutil.copy2(manifest, dst / manifest.name)
print(f'staged in {time.time() - t0:.0f}s')

text_npz = OUT / text_window_probe.TEXT_NPZ
run([sys.executable, '-m', 'core.tools.text_window_probe',
     '--dota-s1-dir', stage / 'DoTA_s1_ncc', '--metadata', DOTA / 'metadata_val.json',
     '--split-file', DOTA / 'val_split.txt',
     '--k-ids-file', K_REPORT / 'k_sources.txt', '--k-manifest', K_REPORT / 'k_sources_manifest.json',
     '--annotation', ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME, '--census', CENSUS,
     '--t2-clip-dir', stage / 'clip_t2', '--video-root', stage / 'video', '--window-root', stage / 'win',
     *(['--text-npz', text_npz] if text_npz.exists() else []),
     '--out-dir', OUT], log=OUT / 'tw_run.log')
print((OUT / text_window_probe.READOUT_MD).read_text())

## 5. Record — then **stop**

Paste `Thesis-V2/outputs/REPORTS/tw_probe/tw_readout.md` back to Claude.

| What the read-out says | Next step |
|---|---|
| **KILL** | Close (ba). The window/text map adds nothing transferable beyond `[x;u;p]` at this bar. |
| **GO**, but `NOT: … beyond the T2 spatial prior` | The gain is *where in the frame*, not text. Not the idea. Decide with the user whether a fixed spatial prior is worth anything. |
| **GO**, `NOT: … beyond multi-scale` | Multi-scale pooling helps, text guidance does not. A cheaper build: uniform windows. |
| **GO** on both sentences, larger on non-ego | The idea holds as written → pre-register a build (an arm, 5 seeds, a fresh test set: DoTA-CAP-eval is already opened). |

Text-weight entropy ≈ 1 means the CLIP text map is flat on these frames, so `text` ≈ `mean` regardless of the
verdict. That is Alert-CLIP's "entangled text" problem, measured.